In [2]:
from google.colab import files
from pathlib import Path, PurePosixPath
import zipfile
import tempfile
import shutil
import sys
import subprocess

print("Select NEW c4-source-colab.zip")
uploaded = files.upload()

names = [n for n in uploaded if n.lower().endswith(".zip")]
if len(names) != 1:
    raise ValueError("Upload exactly one ZIP.")

dest = Path(tempfile.mkdtemp(prefix="c4_updated_", dir="/content"))

with zipfile.ZipFile(names[0]) as archive:
    for member in archive.infolist():
        name = member.filename.replace("\\", "/")
        relative = PurePosixPath(name)

        if (
            relative.is_absolute()
            or ".." in relative.parts
            or any(":" in part for part in relative.parts)
        ):
            raise ValueError(f"Unsafe ZIP path: {name}")

        target = dest.joinpath(*relative.parts)

        if member.is_dir() or name.endswith("/"):
            target.mkdir(parents=True, exist_ok=True)
        else:
            target.parent.mkdir(parents=True, exist_ok=True)
            with archive.open(member) as src, target.open("wb") as dst:
                shutil.copyfileobj(src, dst)

roots = [
    p.parent for p in dest.rglob("requirements-training.txt")
    if (p.parent / "training").is_dir()
]
if len(roots) != 1:
    raise RuntimeError(f"Expected one C4 source folder, found {len(roots)}.")

C4_ROOT = roots[0]
sys.path.insert(0, str(C4_ROOT))

subprocess.check_call([
    sys.executable, "-m", "pip", "install", "-q",
    "-r", str(C4_ROOT / "requirements-training.txt")
])

print("Updated source ready:", C4_ROOT)

Select NEW c4-source-colab.zip


Saving c4-source-colab.zip to c4-source-colab.zip
Updated source ready: /content/c4_updated_s5hs2ipz/ai-services/c4


In [4]:
from google.colab import drive
from training.paths import stage_dirs
from training.config import EnvConfig, TrainConfig
from training.pipeline import load_simulator_from_artifacts
from training.environment import make_env
from gymnasium.utils.env_checker import check_env
import training


OUT = Path("/content/drive/MyDrive/AdaptiveLearnSE/c4_runs")
if not Path("/content/drive/MyDrive").is_dir():
    raise RuntimeError("Drive is not mounted.")

DIRS = stage_dirs(OUT)
ARTIFACTS = DIRS["artifacts"]

for name in ["response_model.joblib", "skill_catalogue.json"]:
    if not (ARTIFACTS / name).is_file():
        raise FileNotFoundError(f"Missing saved artifact: {name}")

if not (DIRS["synthetic"] / "simulator_assumptions.json").is_file():
    raise FileNotFoundError("Notebook 02 simulator artifacts are missing.")

env_cfg = EnvConfig(include_simulation_only_actions=False)
sim = load_simulator_from_artifacts(ARTIFACTS, env_cfg)

env = make_env(sim, env_cfg)
check_env(env, skip_render_check=True)
env.close()

train_cfg = TrainConfig(mode="smoke", device="cpu")

expected = ("dqn", "ppo", "weakest_mastery", "random_valid")
assert train_cfg.resolved().algorithms == expected, (
    "Old configuration loaded. Check the uploaded ZIP and restart."
)

print("Loaded training package:", training.__file__)
print("Environment checks passed.")
print("Comparison methods:", train_cfg.resolved().algorithms)
print("Saved response model reused; no refitting.")

Loaded training package: /content/c4_updated_s5hs2ipz/ai-services/c4/training/__init__.py
Environment checks passed.
Comparison methods: ('dqn', 'ppo', 'weakest_mastery', 'random_valid')
Saved response model reused; no refitting.


In [5]:
from training.train_rl import run_training_comparison
import time

started = time.time()
print("Starting updated smoke comparison...", flush=True)

results = run_training_comparison(
    simulator=sim,
    env_config=env_cfg,
    train_config=train_cfg,
    output_root=OUT,
)

print("Completed in minutes:", round((time.time() - started) / 60, 2))
print("New run folder:", results["run_dir"])
print("Validation selection:", results["selection"])

Starting updated smoke comparison...


Gym has been unmaintained since 2022 and does not support NumPy 2.0 amongst other critical functionality.
Please upgrade to Gymnasium, the maintained drop-in replacement of Gym, or contact the authors of your software and request that they upgrade.
See the migration guide at https://gymnasium.farama.org/introduction/migration_guide/ for additional information.
/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


Completed in minutes: 0.4
New run folder: /content/drive/MyDrive/AdaptiveLearnSE/c4_runs/train_smoke_20261012T035704.286541+0000
Validation selection: {'selected': 'random_valid', 'selected_label': 'Random-valid control (uniform among masked-valid actions; explicitly seeded)', 'mean_val_return_by_algorithm': {'dqn': -8.016000018119811, 'ppo': -39.64200009584427, 'weakest_mastery': 5.780000534653668, 'random_valid': 9.812000058591362}, 'comparison_notes': {'mask_vs_penalty': 'weakest_mastery and random_valid use valid-action masks. Vanilla DQN/PPO learn unavailable actions only through penalties; invalid actions are not silently remapped.', 'pending_recent_performance_baseline': 'A distinct recent-performance baseline required by the research proposal is not implemented yet. random_valid is a secondary control only and must not be described as that baseline.', 'legacy_alias': 'Older smoke runs labelled rule_based and lowest_first used the same weakest-mastery heuristic (alias). New runs

/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


In [6]:
import pandas as pd

rows = []
for seed, methods in results["seeds"].items():
    for method, payload in methods.items():
        metrics = payload["val"]
        rows.append({
            "seed": seed,
            "method": method,
            "mean_return": metrics["return"]["mean"],
            "simulated_mastery_gain": metrics["mastery_gain_sum"]["mean"],
            "invalid_action_rate": metrics["invalid_action_rate"]["mean"],
        })

display(pd.DataFrame(rows))

,seed,method,mean_return,simulated_mastery_gain,invalid_action_rate
0,42,dqn,-8.016000,1.544000,0.296
1,42,ppo,-39.642000,0.388000,0.828
2,42,weakest_mastery,5.780001,1.780001,0.000
3,42,random_valid,9.812000,2.212000,0.000


/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


In [7]:
import time
from training.config import TrainConfig
from training.train_rl import run_training_comparison

full_cfg = TrainConfig(mode="full", device="cpu")

print("Full training configuration:")
print(full_cfg.resolved())
print("\nStarting full DQN/PPO training...")

started = time.time()

full_results = run_training_comparison(
    simulator=sim,
    env_config=env_cfg,
    train_config=full_cfg,
    output_root=OUT,
)

print("\nCompleted in minutes:", round((time.time() - started) / 60, 2))
print("Run folder:", full_results["run_dir"])
print("Validation selection:", full_results["selection"])

Full training configuration:
TrainConfig(mode='full', seeds=(0, 1, 2), timesteps=100000, device='cpu', checkpoint_freq=1000, eval_episodes=20, algorithms=('dqn', 'ppo', 'weakest_mastery', 'random_valid'))

Starting full DQN/PPO training...


Streaming output truncated to the last 5000 lines.
/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replac


Completed in minutes: 18.41
Run folder: /content/drive/MyDrive/AdaptiveLearnSE/c4_runs/train_full_20261012T040616.853409+0000
Validation selection: {'selected': 'random_valid', 'selected_label': 'Random-valid control (uniform among masked-valid actions; explicitly seeded)', 'mean_val_return_by_algorithm': {'dqn': -7.026833423326413, 'ppo': -38.763166741480426, 'weakest_mastery': 7.22666721592347, 'random_valid': 9.858666707575319}, 'comparison_notes': {'mask_vs_penalty': 'weakest_mastery and random_valid use valid-action masks. Vanilla DQN/PPO learn unavailable actions only through penalties; invalid actions are not silently remapped.', 'pending_recent_performance_baseline': 'A distinct recent-performance baseline required by the research proposal is not implemented yet. random_valid is a secondary control only and must not be described as that baseline.', 'legacy_alias': 'Older smoke runs labelled rule_based and lowest_first used the same weakest-mastery heuristic (alias). New runs r

/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


In [10]:
import pandas as pd
from IPython.display import display

rows = []

for seed, methods in full_results["seeds"].items():
    for method, payload in methods.items():
        metrics = payload["val"]

        rows.append({
            "seed": seed,
            "method": method,
            "mean_return": metrics["return"]["mean"],
            "simulated_mastery_gain": metrics["mastery_gain_sum"]["mean"],
            "invalid_action_rate": metrics["invalid_action_rate"]["mean"],
        })

full_table = pd.DataFrame(rows)

print("Results per seed:")
display(full_table)

print("Mean and standard deviation across seeds:")
display(
    full_table.groupby("method")[
        ["mean_return", "simulated_mastery_gain", "invalid_action_rate"]
    ].agg(["mean", "std"])
)

print("\nValidation selection:")
print(full_results["selection"])

print("\nSaved run folder:")
print(full_results["run_dir"])

Results per seed:


/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


,seed,method,mean_return,simulated_mastery_gain,invalid_action_rate
0,0,dqn,-0.861000,1.849000,0.186
1,0,ppo,-38.869500,0.433000,0.819
2,0,weakest_mastery,7.264001,1.939001,0.000
3,0,random_valid,9.532000,2.182000,0.000
4,1,dqn,-13.122000,1.383000,0.388
5,1,ppo,-39.015000,0.430000,0.822
6,1,weakest_mastery,7.320001,1.945001,0.000
7,1,random_valid,10.120000,2.245000,0.000
8,2,dqn,-7.097500,1.655000,0.299
9,2,ppo,-38.405000,0.445000,0.810


Mean and standard deviation across seeds:


/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


mean_return           simulated_mastery_gain            \
                       mean       std                   mean       std   
method                                                                   
dqn               -7.026833  6.130805               1.629000  0.234085   
ppo              -38.763167  0.318599               0.436000  0.007937   
random_valid       9.858667  0.299395               2.217000  0.032078   
weakest_mastery    7.226667  0.116573               1.935001  0.012490   

                invalid_action_rate            
                               mean       std  
method                                         
dqn                           0.291  0.101237  
ppo                           0.817  0.006245  
random_valid                  0.000  0.000000  
weakest_mastery               0.000  0.000000


Validation selection:
{'selected': 'random_valid', 'selected_label': 'Random-valid control (uniform among masked-valid actions; explicitly seeded)', 'mean_val_return_by_algorithm': {'dqn': -7.026833423326413, 'ppo': -38.763166741480426, 'weakest_mastery': 7.22666721592347, 'random_valid': 9.858666707575319}, 'comparison_notes': {'mask_vs_penalty': 'weakest_mastery and random_valid use valid-action masks. Vanilla DQN/PPO learn unavailable actions only through penalties; invalid actions are not silently remapped.', 'pending_recent_performance_baseline': 'A distinct recent-performance baseline required by the research proposal is not implemented yet. random_valid is a secondary control only and must not be described as that baseline.', 'legacy_alias': 'Older smoke runs labelled rule_based and lowest_first used the same weakest-mastery heuristic (alias). New runs report weakest_mastery once.'}, 'note': 'Selection uses validation only; do not tune on final test.'}

Saved run folder:
/conte